# Llama Document-Aware Extraction Demo

## Simplified YAML-First Architecture with Document-Specific Field Filtering

This notebook demonstrates the simplified `llama_document_aware.py` system with **YAML-first document detection** and **document-aware field filtering** that intelligently selects relevant fields based on document type.

### 📋 Complete Semantically Grouped Field Schema (49 fields)
The system supports extraction of these fields from `evaluation_data/ground_truth.csv`:

**Document Identification:**
`DOCUMENT_TYPE`, `INVOICE_NUMBER`, `RECEIPT_NUMBER`

**Business Details:**
`SUPPLIER_NAME`, `BUSINESS_ABN`, `BUSINESS_ADDRESS`, `BUSINESS_PHONE`, `SUPPLIER_WEBSITE`, `SUPPLIER_EMAIL`

**Customer Information:**
`PAYER_NAME`, `PAYER_ABN`, `PAYER_ADDRESS`, `PAYER_PHONE`, `PAYER_EMAIL`

**Temporal Data:**
`INVOICE_DATE`, `DUE_DATE`, `TRANSACTION_DATE`, `STATEMENT_DATE_RANGE`, `CREDIT_CARD_DUE_DATE`

**Line Items:**
`LINE_ITEM_DESCRIPTIONS`, `LINE_ITEM_QUANTITIES`, `LINE_ITEM_PRICES`, `LINE_ITEM_TOTAL_PRICES`, `LINE_ITEM_GST_AMOUNTS`, `LINE_ITEM_DISCOUNT_AMOUNTS`

**Financial Totals:**
`SUBTOTAL_AMOUNT`, `TOTAL_DISCOUNT_AMOUNT`, `GST_AMOUNT`, `IS_GST_INCLUDED`, `TOTAL_AMOUNT`, `TOTAL_AMOUNT_PAID`, `BALANCE_OF_PAYMENT`, `TOTAL_AMOUNT_PAYABLE`

**Payment & Location:**
`PAYMENT_METHOD`, `STORE_LOCATION`

**Banking Information:**
`BANK_NAME`, `BANK_BSB_NUMBER`, `BANK_ACCOUNT_NUMBER`, `BANK_ACCOUNT_HOLDER`, `ACCOUNT_OPENING_BALANCE`, `ACCOUNT_CLOSING_BALANCE`

**Transactions:**
`TOTAL_CREDITS`, `TOTAL_DEBITS`, `NET_TRANSACTION_AMOUNT`, `TRANSACTION_DATES`, `TRANSACTION_AMOUNTS_PAID`, `TRANSACTION_AMOUNTS_RECEIVED`, `TRANSACTION_BALANCES`

### 🎯 Document-Aware Field Filtering

**Key Innovation:** The system uses **simplified YAML-first document detection** and intelligently filters to document-specific subsets:

- **📄 Invoice Documents**: 29 relevant fields (41% reduction)
- **🧾 Receipt Documents**: 20 relevant fields (59% reduction)  
- **🏦 Bank Statements**: 16 relevant fields (67% reduction)

This targeted approach delivers:
- ⚡ **Faster processing** with fewer fields to extract
- 🎯 **Higher accuracy** with document-specific prompts
- 💡 **Better resource utilization** focusing on relevant data
- 📈 **Improved performance** through YAML-first prompt configuration
- 🧹 **Cleaner architecture** with 91% less detection code

---

In [ ]:
# Import required libraries
import sys
import os
from pathlib import Path
from IPython.display import Image, display, HTML
import time

# Set project root (notebook is now in project root)
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Import document-aware system
from llama_document_aware import DocumentAwareLlamaHandler
from common.evaluation_metrics import load_ground_truth

print("📚 Libraries imported successfully")
print(f"🗂️ Project root: {project_root}")
print(f"📍 Current directory: {Path.cwd()}")

## Configuration

Set up the document-aware processor with your model path:

In [ ]:
# Configuration - UPDATE WITH YOUR MODEL PATH
# MODEL_PATH = "/efs/shared/PTM/Llama-3.2-11B-Vision-Instruct"
MODEL_PATH = "/home/jovyan/nfs_share/models/Llama-3.2-11B-Vision-Instruct"

# TEST_IMAGE = "evaluation_data/image_042.png"  # INVOICE example
# TEST_IMAGE = "evaluation_data/image_005.png"  # RECEIPT example
# TEST_IMAGE = "evaluation_data/image_004.png"  # RECEIPT example
# TEST_IMAGE = "evaluation_data/image_004.png"  # RECEIPT example
TEST_IMAGE = "evaluation_data/image_006.png"  # Bank statement example
# TEST_IMAGE = "evaluation_data/image_041.png"  # NON_STANDARD COMPLEX example


GROUND_TRUTH_PATH = "evaluation_data/ground_truth.csv"

# Initialize the document-aware handler
print("🚀 Initializing Document-Aware Llama Handler...")
handler = DocumentAwareLlamaHandler(MODEL_PATH, debug=True)
print("✅ Handler initialized successfully")

## Demo Image

Let's examine the test document we'll be processing:

In [ ]:
# Display the test image
image_path = project_root / TEST_IMAGE
if image_path.exists():
    display(Image(str(image_path), width=800))
    print(f"📄 Document: {image_path.name}")
else:
    print(f"❌ Image not found: {image_path}")
    print("Please update the TEST_IMAGE path above")

# Step 1: Detect document type and get schema
print("🔍 STEP 1: YAML-First Document Type Detection")
print("=" * 50)

# Time the document type detection
detection_start = time.perf_counter()
classification_info = handler.detect_and_classify_document(str(image_path))
detection_time = time.perf_counter() - detection_start

print(f"\n📋 Document Type: {classification_info['document_type']}")
print(f"📊 Field Count: {classification_info['field_count']} fields (vs 49 universal)")
print(f"⚡ Efficiency: {(49 - classification_info['field_count'])/49*100:.0f}% fewer fields than universal approach")
print(f"⏱️ Detection Time: {detection_time:.3f}s")
print(f"🎯 Method: Simplified YAML-first detection (91% less code)")
print(f"\n🎯 Target Fields: {', '.join(classification_info['field_names'][:5])}...")

# Store timing for final summary
step1_time = detection_time

In [ ]:
# Step 1: Detect document type and get schema
print("🔍 STEP 1: Document Type Detection")
print("=" * 50)

# Time the document type detection
detection_start = time.perf_counter()
classification_info = handler.detect_and_classify_document(str(image_path))
detection_time = time.perf_counter() - detection_start

print(f"\n📋 Document Type: {classification_info['document_type']}")
print(f"📊 Field Count: {classification_info['field_count']} fields (vs 49 universal)")
print(f"⚡ Efficiency: {(49 - classification_info['field_count'])/49*100:.0f}% fewer fields than universal approach")
print(f"⏱️ Detection Time: {detection_time:.3f}s")
print(f"\n🎯 Target Fields: {', '.join(classification_info['field_names'][:5])}...")

# Store timing for final summary
step1_time = detection_time

## Step 2: YAML-First Prompt Generation

The system generates a high-performance prompt using YAML configuration:

In [ ]:
# Step 2: Show the YAML-first prompt generation
print("📝 STEP 2: YAML-First Prompt Generation")
print("=" * 50)

# Time the prompt generation
prompt_start = time.perf_counter()

# Access the processor to show the prompt (for demo purposes)
if handler.processor:
    # Reconfigure processor for the detected fields
    handler.processor.field_list = classification_info['field_names']
    handler.processor.field_count = len(classification_info['field_names'])
    
    # Generate the prompt
    prompt = handler.processor.generate_dynamic_prompt()
    prompt_time = time.perf_counter() - prompt_start
    
    print(f"🎯 Generated prompt for {len(classification_info['field_names'])} fields")
    print(f"📏 Prompt length: {len(prompt)} characters")
    print(f"⏱️ Prompt Generation Time: {prompt_time:.6f}s (microsecond precision)")
    print("\n🔍 COMPLETE YAML-FIRST PROMPT:")
    print("-" * 80)
    # Show the COMPLETE prompt without truncation
    print(prompt)
    print("-" * 80)
    
    # Store timing for final summary
    step2_time = prompt_time
else:
    print("⚠️ Processor not initialized yet")
    step2_time = 0.0

## Step 3: Document-Aware Extraction

Process the document with type-specific field extraction:

In [ ]:
# Step 3: Extract with document-aware processing
print("🔍 STEP 3: Document-Aware Extraction")
print("=" * 50)

# Time the full extraction process
extraction_start = time.perf_counter()

# Time just the model processing part
model_start = time.perf_counter()
result = handler.process_document_aware(str(image_path), classification_info)
model_end = time.perf_counter()

extraction_total_time = time.perf_counter() - extraction_start

print("\n⏱️ TIMING BREAKDOWN:")
print(f"   🤖 Model Processing: {result['processing_time']:.3f}s")
print(f"   📊 Total Step Time: {extraction_total_time:.3f}s")
print(f"   💾 Overhead (parsing/formatting): {extraction_total_time - result['processing_time']:.3f}s")

print("\n📊 EXTRACTION RESULTS:")
print(f"   ✅ Fields Found: {result['detected_fields']}/{result['total_fields']}")
print(f"   📈 Field Coverage: {result['detected_fields']/result['total_fields']*100:.1f}%")
print(f"   ⚡ Fields/Second: {result['detected_fields']/result['processing_time']:.1f} fields/s")

# Store timing for final summary
step3_time = extraction_total_time
model_inference_time = result['processing_time']

## Step 4: Extracted Data Visualization

Display the extracted data with visual status indicators:

In [ ]:
# Step 4: Display extracted data with ground truth comparison
print("📊 STEP 4: Extracted Data Results with Ground Truth Comparison")
print("=" * 100)

# Load ground truth for comparison
ground_truth_data = {}
ground_truth_path = project_root / GROUND_TRUTH_PATH
if ground_truth_path.exists():
    all_ground_truth = load_ground_truth(str(ground_truth_path))
    image_name = Path(image_path).name
    if image_name in all_ground_truth:
        ground_truth_data = all_ground_truth[image_name]
        print(f"✅ Ground truth loaded for {image_name}")
    else:
        print(f"⚠️ No ground truth available for {image_name}")
else:
    print(f"⚠️ Ground truth file not found: {ground_truth_path}")

print("-" * 100)

# Display header
print(f"{'STATUS':<8} {'FIELD':<30} {'EXTRACTED':<35} {'GROUND TRUTH':<35}")
print("=" * 100)

extracted_data = result["extracted_data"]
found_count = 0
total_count = len(extracted_data)
match_count = 0

for field_name, extracted_value in extracted_data.items():
    # Get ground truth value
    gt_value = ground_truth_data.get(field_name, "NOT_AVAILABLE")
    
    # Determine extraction status
    if extracted_value != "NOT_FOUND":
        extraction_status = "✅"
        found_count += 1
    else:
        extraction_status = "❌"
    
    # Check if extracted matches ground truth (case-insensitive comparison)
    extracted_clean = str(extracted_value).strip().lower()
    gt_clean = str(gt_value).strip().lower()
    
    # Handle some special cases for matching
    if extracted_clean == gt_clean:
        match_indicator = "✓"
        match_count += 1
    elif extracted_value == "NOT_FOUND" and (gt_value == "NOT_FOUND" or gt_value == "NOT_AVAILABLE" or gt_value == ""):
        match_indicator = "✓"
        match_count += 1
    elif extracted_value != "NOT_FOUND" and gt_value != "NOT_FOUND" and gt_value != "NOT_AVAILABLE":
        # Partial match check (e.g., for amounts with different formatting)
        if any(x in extracted_clean for x in gt_clean.replace("$", "").replace(",", "").split()):
            match_indicator = "≈"  # Partial match
        else:
            match_indicator = "✗"
    else:
        match_indicator = "✗"
    
    # Format the output - truncate long values for display
    extracted_display = (extracted_value[:32] + "...") if len(str(extracted_value)) > 35 else extracted_value
    gt_display = (gt_value[:32] + "...") if len(str(gt_value)) > 35 else gt_value
    
    # Color coding based on match status
    if match_indicator == "✓":
        print(f"{extraction_status:<8} {field_name:<30} {extracted_display:<35} {gt_display:<35} {match_indicator}")
    elif match_indicator == "≈":
        print(f"{extraction_status:<8} {field_name:<30} {extracted_display:<35} {gt_display:<35} {match_indicator}")
    else:
        print(f"{extraction_status:<8} {field_name:<30} {extracted_display:<35} {gt_display:<35} {match_indicator}")

print("\n" + "=" * 100)
print("📈 EXTRACTION SUMMARY:")
print(f"   ✅ Fields Found: {found_count}/{total_count} ({found_count/total_count*100:.1f}%)")
print(f"   🎯 Exact Matches: {match_count}/{total_count} ({match_count/total_count*100:.1f}%)")
print(f"   📊 Extraction Success Rate: {found_count/total_count*100:.1f}%")
print(f"   🏆 Accuracy (matches/found): {match_count/found_count*100:.1f}%" if found_count > 0 else "   🏆 Accuracy: N/A")
print(f"   ⏱️ Processing Time: {result['processing_time']:.3f}s")
print(f"   🎯 Document Type: {result['document_type']}")
print(f"   ⚡ Field Reduction: {(49-total_count)/49*100:.0f}% fewer fields than universal")
print(f"   💾 Model: Llama-3.2-11B-Vision-Instruct")

print("\n📋 LEGEND:")
print("   ✓ = Exact match")
print("   ≈ = Partial match")
print("   ✗ = No match")
print("   NOT_AVAILABLE = Field not present in ground truth")

## Step 5: Ground Truth Evaluation

Compare against ground truth data for accuracy measurement:

In [ ]:
# Step 5: Evaluate against ground truth
print("📊 STEP 5: Ground Truth Evaluation")
print("=" * 50)

# Time the evaluation process
evaluation_start = time.perf_counter()

# Load ground truth
ground_truth_path = project_root / GROUND_TRUTH_PATH
if ground_truth_path.exists():
    ground_truth = load_ground_truth(str(ground_truth_path))
    image_name = Path(image_path).name
    
    if image_name in ground_truth:
        # Evaluate with document-type-specific metrics
        evaluation_report = handler.evaluate_document_aware([result], ground_truth)
        evaluation_time = time.perf_counter() - evaluation_start
        
        # Extract metrics for this document type
        doc_type = result['document_type']
        type_metrics = evaluation_report['by_document_type'].get(doc_type, {})
        
        print("\n📈 ACCURACY RESULTS:")
        print(f"   🎯 Overall Accuracy: {type_metrics.get('avg_accuracy', 0)*100:.1f}%")
        print(f"   ✅ Meets Threshold: {'Yes' if type_metrics.get('meets_threshold', 0) > 0 else 'No'}")
        print(f"   🔥 Critical Fields Perfect: {'Yes' if type_metrics.get('critical_perfect', 0) > 0 else 'No'}")
        
        if doc_type == 'invoice':
            ato_compliant = type_metrics.get('ato_compliant', 0) > 0
            print(f"   🏛️ ATO Compliant: {'Yes' if ato_compliant else 'No'}")
        
        print("\n⏱️ EVALUATION TIMING:")
        print(f"   📊 Evaluation Time: {evaluation_time:.3f}s")
        print(f"   📈 Fields/Second (eval): {result['detected_fields']/evaluation_time:.1f} comparisons/s")
        
        print("\n📊 PERFORMANCE COMPARISON:")
        print(f"   🆚 Document-Aware: {type_metrics.get('avg_accuracy', 0)*100:.1f}% accuracy")
        print(f"   📉 Field Reduction: {(49-total_count)/49*100:.0f}% fewer fields")
        print(f"   ⚡ Processing Speed: {result['detected_fields']/model_inference_time:.1f} fields/s")
        
        # Store timing for final summary
        step5_time = evaluation_time
        
    else:
        print(f"⚠️ No ground truth available for {image_name}")
        step5_time = time.perf_counter() - evaluation_start
else:
    print(f"❌ Ground truth file not found: {ground_truth_path}")
    step5_time = time.perf_counter() - evaluation_start

In [ ]:
## ⏱️ Comprehensive Timing Analysis (V100 Production Performance)

# Calculate total pipeline time
total_pipeline_time = step1_time + step2_time + step3_time + step5_time

print("🎯 PRODUCTION PERFORMANCE METRICS")
print("=" * 70)
print(f"📊 Document: {Path(image_path).name} ({result['document_type']})")
print(f"🔧 Hardware: V100 GPU (production target)")
print(f"📈 Field Coverage: {result['detected_fields']}/{result['total_fields']} fields")
print()

print("⏱️  DETAILED TIMING BREAKDOWN:")
print("-" * 70)
print(f"Step 1 - Document Detection:     {step1_time:8.3f}s ({step1_time/total_pipeline_time*100:5.1f}%)")
print(f"Step 2 - YAML Prompt Generation: {step2_time:8.6f}s ({step2_time/total_pipeline_time*100:5.1f}%)")
print(f"Step 3 - Model Inference:        {model_inference_time:8.3f}s ({model_inference_time/total_pipeline_time*100:5.1f}%)")
print(f"Step 4 - Result Processing:      {step3_time - model_inference_time:8.3f}s ({(step3_time - model_inference_time)/total_pipeline_time*100:5.1f}%)")
print(f"Step 5 - Accuracy Evaluation:    {step5_time:8.3f}s ({step5_time/total_pipeline_time*100:5.1f}%)")
print("-" * 70)
print(f"TOTAL PIPELINE TIME:             {total_pipeline_time:8.3f}s (100.0%)")
print()

print("🚀 PRODUCTION THROUGHPUT METRICS:")
print("-" * 70)
print(f"🎯 Fields per Second:            {result['detected_fields']/model_inference_time:8.1f} fields/s")
print(f"📄 Documents per Minute:          {60/total_pipeline_time:8.1f} docs/min")
print(f"📊 Documents per Hour:            {3600/total_pipeline_time:8.0f} docs/hour")
print()

print("⚡ EFFICIENCY GAINS (vs Universal 49-field approach):")
print("-" * 70)
universal_estimate = model_inference_time * (49/result['total_fields'])  # Estimated time for 49 fields
time_savings = universal_estimate - model_inference_time
print(f"🔄 Universal (49 fields) est:    {universal_estimate:8.3f}s")
print(f"🎯 Document-Aware actual:        {model_inference_time:8.3f}s")
print(f"💡 Time Savings:                 {time_savings:8.3f}s ({time_savings/universal_estimate*100:5.1f}% faster)")
print(f"⚡ Efficiency Gain:              {(49/result['total_fields']):8.1f}x speedup potential")
print()

print("🏆 KEY PRODUCTION METRICS:")
print("-" * 70)
accuracy = type_metrics.get('avg_accuracy', 0) * 100
print(f"🎯 Accuracy:                     {accuracy:8.1f}%")
print(f"⚡ Processing Speed:             {model_inference_time:8.3f}s per document")
print(f"💾 Memory Efficiency:           {result['total_fields']:8d} fields (vs 49 universal)")
print(f"🔧 V100 Optimization:            ✅ 8-bit quantization enabled")
print(f"📋 YAML-First Config:            ✅ Maintainable prompts")
print()

# Production readiness assessment
if accuracy >= 95.0 and model_inference_time <= 45.0:  # 45s threshold for production
    status = "✅ PRODUCTION READY"
elif accuracy >= 90.0 and model_inference_time <= 60.0:
    status = "⚠️  PRODUCTION VIABLE"
else:
    status = "❌ NEEDS OPTIMIZATION"

print(f"🏭 PRODUCTION READINESS:          {status}")

# Estimate daily throughput
daily_docs = int(3600 * 8 / total_pipeline_time)  # 8 hour work day
print(f"📈 Daily Throughput (8hrs):       {daily_docs:8d} documents/day")
print("=" * 70)